# Anomaly Detection

*Real-world Anomaly Detection in Surveillance Videos (Sultani, 2018)*

The goal of anomaly detection is to identify activities that deviate from normal behavior and to determine the time window in which the anomaly occurs.

Anomaly detection can be viewed as a **coarse-level video understanding task** because it only distinguishes between **normal** and **abnormal** events. Its purpose is to filter out anomalous activities from regular patterns.

After an anomaly has been detected, a **classification** system can be applied to determine the specific type of activity (e.g., accident, theft, fighting, or falling).

**Pipeline**

1. Detect whether an event is normal or anomalous.
2. Locate the time interval of the anomaly.
3. Classify the anomaly into a specific activity category.

**Example**

- Anomaly Detection: "Something unusual happened between 10:05 and 10:07."
- Classification: "The unusual event was a fight."

## Multiple Instance Problem in Video Anomaly Detection

In weakly supervised anomaly detection, videos are labeled only at the video level rather than at the frame level.

A video is treated as a **bag** containing multiple temporal segments (**instances**).

- Normal video: all instances are normal.
- Anomalous video: at least one instance is anomalous.

The exact anomalous segment is unknown during training, making the task a **Multiple Instance Learning (MIL)** problem.

MIL allows models to learn anomaly localization using only video-level labels, avoiding the expensive process of frame-level annotation.

## Anomaly Ranking Model

Using weakly labeled training videos, the model learns an **anomaly ranking function** that assigns higher anomaly scores to anomalous video segments than to normal ones.

During inference, an untrimmed video is divided into multiple segments. Each segment is passed through the network, which outputs an anomaly score indicating how abnormal that segment is. Segments with high anomaly scores are identified as potential anomalies.

## Main Contributions

1. Proposed a **Multiple Instance Learning (MIL)** approach for video anomaly detection using only weakly labeled videos (normal/anomalous video-level labels) instead of costly segment-level annotations.

2. Introduced a **MIL ranking loss** with **sparsity** and **smoothness** constraints to learn anomaly scores for video segments and localize anomalies without frame-level annotations.

## Architecture

<div>
    <img src='../images/AnomalyArch.png' width="1000">
</div>

## Method

The proposed approach (summarized in Figure 1) begins with dividing surveillance videos into a ﬁxed number of segments during training. These segments make instances in a bag. Using both positive (anomalous) and negative (normal) bags, we train the anomaly detection model using the proposed deep MIL ranking loss.

### Multiple Instance Learning

In the context of supervised anomaly detection, a classifier needs temporal annotations of each segment in videos. However, obtaining temporal
annotations for videos is time consuming and laborious. In MIL, precise temporal locations of anomalous events in videos are unknown. Instead, only
video-level labels indicating the presence of an anomaly in the whole video is needed. 

A video containing anomalies is labeled as positive and a video without any anomaly is labeled as negative. Then, we represent a positive video as
a positive bag $B_a$, where different temporal segments make individual instances in the bag, $(p_1, p_2, . . . , p_m)$, where $m$ is the number of instances in the bag. We assume that **at least one of these instances contains the anomaly**.

Similarly, the negative video is denoted by a negative bag, $B_n$, where temporal segments in this bag form negative instances $(n_1, n_2, . . . , n_m)$. In the **negative bag, none of the instances contain an anomaly.**

Since the exact information (i.e. instance-level label) of the positive instances is unknown, one can optimize the objective function with respect to the maximum scored instance in each bag (assuming the responsible for the bag label is the instance with the highest score):

Hence we pick the instance with the highest score:

The highest-scoring instance within a bag is defined as:

$$
\max_{i \in B_j} \left( w \cdot \phi(x_i) - b \right)
$$

The complete MIL objective function is given by:

$$
\min_{w}
\frac{1}{z}
\sum_{j=1}^{z}
\max \left(
0,\,
1 - Y_{B_j}
\left(
\max_{i \in B_j}
\left( w \cdot \phi(x_i) - b \right)
\right)
\right)
+ \lVert w \rVert^2
$$

where $Y_{B_j}$ denotes bag-level label, $z$ is the total number of bags, and all the other variables are the same as in Hinge loss equation.

### Deep MIL Ranking Model

Anomalous behavior is difficult to define accurately , since it is quite subjective and can vary largely from person to person. Further, it is not obvious how to assign 1/0 labels to anomalies. Moreover, due to the unavailability of sufficient examples of anomaly, **anomaly detection is usually treated as low-likelihood pattern detection instead of classification problem**.

**Low-likelihood patterns** are observations or behaviors that have a low probability of occurring under the distribution of normal data, commonly denoted as $P(x)$. In anomaly detection, events that frequently occur are associated with higher probability values, while rare or unusual events have lower probabilities. Therefore, patterns with low $P(x)$ are often considered potential anomalies because they deviate from the expected behavior learned from the data.

For example, in a surveillance system monitoring a pedestrian area, people walking normally would correspond to a high-probability event (high $P(x)$) because it occurs frequently. In contrast, a vehicle entering the pedestrian zone would represent a low-probability event (low $P(x)$), making it a potential anomaly due to its rarity and deviation from normal scene activity.

Ideally, a ranking loss which encourages high scores for anomalous video segments as compared to normal segments, would be:

$$
f(V_a) > f(V_n)
$$

where $V_a$ and $V_n$ represent anomalous and normal video segments, $f(V_a)$ and $f(V_n)$ represent the corresponding predicted scores, respectively. The above ranking function should work well if the segment-level annotations are known during training

However, since there are no video segment-level annotations, it is not possible to use the original hinge loss formulation directly. Hence, the following Multiple Instance Learning (MIL) ranking objective is defined:

$$
\max_{i \in B_a} f(V_a^i)
>
\max_{i \in B_n} f(V_n^i)
$$

where the maximum is taken over all video segments in each bag. In other words, the segment with the highest anomaly score in the positive bag should have a higher score than the segment with the highest anomaly score in the negative bag.

With this formulation, the focus is to enforce ranking only on the two instances having the highest anomaly scores in the positive and negative bags, rather than on every instance.

Under the Multiple Instance Learning assumption, a positive bag contains at least one anomalous instance. Therefore, the instance with the highest anomaly score is used as a surrogate for the unknown positive instance.

Thus:

- The segment corresponding to the highest anomaly score in the positive bag is most likely the true positive instance (an anomalous segment).
- The segment corresponding to the highest anomaly score in the negative bag is the one that looks most similar to an anomalous segment but is actually a normal instance.

The objective is therefore to push the positive and negative instances farther apart in terms of anomaly score.

Let:

$$
s_a = \max_{i \in B_a} f(V_a^i)
$$

$$
s_n = \max_{i \in B_n} f(V_n^i)
$$

At a high level, we want the anomaly score of the positive bag to be larger than that of the negative bag. More specifically, we require the difference between both scores to be at least 1:

$$
s_a - s_n \ge 1
$$

Recall that the hinge loss function is defined as:

$$
L(x,y)=\max(0,\;1-yf(x))
$$

Replacing the classification term $yf(x)$ with the ranking term $(s_a - s_n)$, the hinge loss becomes:

$$
l(B_a, B_n)
=
\max\!\left(
0,\;
1
-
(s_a - s_n)
\right)
$$

Expanding the expression:

$$
l(B_a, B_n)
=
\max\!\left(
0,\;
1
-
s_a + s_n
\right)
$$

Substituting the definitions of $s_a$ and $s_n$:

$$
l(B_a, B_n)
=
\max\!\left(
0,\;
1
-
\max_{i \in B_a} f(V_a^i)
+
\max_{i \in B_n} f(V_n^i)
\right)
$$

This formulation can be interpreted as a pairwise ranking hinge loss, where the objective is to ensure that the most anomalous segment in a positive bag receives a higher anomaly score than the most suspicious segment in a negative bag by a margin of at least one.

Anomalies typically occur only in a small number of segments within a video. Therefore, a sparsity constraint is introduced to prevent the model from assigning high anomaly scores to all segments. This encourages the network to highlight only the truly anomalous portions of the video.

Additionally, anomalies usually persist across consecutive segments rather than appearing as isolated spikes. Therefore, a smoothness constraint is applied to encourage neighboring segments to have similar anomaly scores, resulting in a more temporally consistent anomaly prediction.

By incorporating the sparsity and smoothness constraints on the instance scores, the loss function becomes:

$$
l(B_a,B_n)
=
\max\left(
0,
1
-
\max_{i \in B_a} f(V_a^i)
+
\max_{i \in B_n} f(V_n^i)
\right)
+
\lambda_1
\sum_{i=1}^{n-1}
\left(
f(V_a^i)-f(V_a^{i+1})
\right)^2
+
\lambda_2
\sum_{i=1}^{n}
f(V_a^i)
$$

where the first summation term represents the temporal smoothness constraint, while the second summation term represents the sparsity constraint.

In this MIL ranking loss, the error is back-propagated from the maximum scored video segments in both positive and negative bags. By training on a large number of positive and negative bags, we expect that the network will learn a generalized model to predict high scores for anomalous segments in positive bags.

Finally, our complete objective function is given by

$$
L(W) = l(B_a, B_n) + \|W\|_F
$$

where $W$ represents model weights.

**Bags Formations**. We divide each video into the equal number of non-overlapping temporal segments and use these video segments as bag instances. Given each video segment, we extract the 3D convolution features. We use this feature representation due to its computational efficiency, the evident capability of capturing appearance and motion dynamics in video action recognition.

## UCF-Crime

A new dataset is introduced in the paper. It contains:

- 1,900 real-world surveillance videos (950 anomalous and 950 normal).
- 13 anomaly categories
    - Abuse, Arrest, Arson, Assault, Accident, Burglary, Explosion, Fighting, Robbery, Shooting, Stealing, Shoplifting, and Vandalism.
- Weak annotations for training (normal/anomalous labels only at the video level).
- Temporal annotations for evaluation.

The videos were collected from YouTube, and the creation of the dataset required several months of manual effort.


## Implementations details

### Feature Extraction
- Used **C3D FC6 features** (4096-dimensional).
- Resized frames to **240 × 320** pixels.
- Fixed frame rate at **30 fps**.
- Extracted features from every **16-frame clip**.
- Applied **L2 normalization** to clip features.
- Computed segment features by averaging all clip features within each segment.

### Neural Network Architecture
- Input: **4096D C3D feature vector**.
- 3 fully connected layers:
  - FC1: **512 units**
  - FC2: **32 units**
  - FC3: **1 output unit**
- Applied **60% dropout** between FC layers.
- Deeper architectures were tested but did **not improve performance**.

### Activations & Optimization
- **ReLU** activation for hidden layers.
- **Sigmoid** activation for the output layer.
- **Adagrad** optimizer.
- Initial learning rate: **0.001**.

### MIL Ranking Loss
- Added sparsity and smoothness constraints.
- Best-performing parameters:
  - λ₁ = λ₂ = **8 × 10⁻⁵**

### Video Segmentation
- Divided each video into **32 non-overlapping temporal segments**.
- Treated each segment as an **instance within a MIL bag**.
- Multi-scale overlapping segments were evaluated but showed **no accuracy improvement**.

### Training Setup
- Mini-batch composition:
  - **30 positive bags**
  - **30 negative bags**
- Gradients computed using **reverse-mode automatic differentiation** in Theano.
- Final gradients obtained through the **chain rule** on the computation graph.

### Training Pipeline
1. Divide each video into **32 segments**.
2. Extract and average **C3D FC6 features** for each segment.
3. Feed segment features into the FC network.
4. Generate an anomaly score for every segment.
5. Compute the MIL ranking loss with sparsity and smoothness constraints.
6. Backpropagate the loss across the entire batch to update model parameters.

## Results

### Comparison with other methods

<div>
    <img src='../images/AnomalyResults.png' width="500">
</div>

### Examples

<div>
    <img src='../images/AnomalyExamples.png' width="1200">
</div>